In [7]:
import torch
from pycocotools.coco import COCO
import os, cv2
import numpy as np

class CocoSegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_path, transforms=None):
        self.coco = COCO(ann_path)
        self.img_dir = img_dir
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.transforms = transforms

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        path = img_info['file_name']
        img = cv2.imread(os.path.join(self.img_dir, path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for ann in anns:
            rle = self.coco.annToMask(ann)
            mask = np.maximum(mask, rle * ann['category_id'])

        if self.transforms:
            img = self.transforms(img)
        mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
        return img, torch.tensor(mask, dtype=torch.long)


    def __len__(self):
        return len(self.ids)


In [12]:
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torch.utils.data import Subset

img_dir = "same-object-images"
ann_path = "same-object-coco.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = CocoSegmentationDataset(img_dir, ann_path, transforms=transforms)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

model = torchvision.models.segmentation.deeplabv3_resnet101(pretrained=False, num_classes=2)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(1):
    model.train()
    running_loss = 0
    print(f"\nEpoch {epoch+1} starting...")

    for i, (imgs, masks) in enumerate(loader):
        imgs, masks = imgs.to(device), masks.to(device)
        output = model(imgs)['out']
        loss = criterion(output, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 10 == 0:
            print(f"  Batch {i}/{len(loader)} - Loss: {loss.item():.4f}")

    print(f"Epoch {epoch+1} done! Avg Loss: {running_loss / len(loader):.4f}")



loading annotations into memory...
Done (t=1.64s)
creating index...
index created!

Epoch 1 starting...
  Batch 0/831 - Loss: 0.9264
  Batch 10/831 - Loss: 0.4806
  Batch 20/831 - Loss: 0.2212
  Batch 30/831 - Loss: 0.1659
  Batch 40/831 - Loss: 0.1466
  Batch 50/831 - Loss: 0.1530
  Batch 60/831 - Loss: 0.0929
  Batch 70/831 - Loss: 0.1031
  Batch 80/831 - Loss: 0.0798
  Batch 90/831 - Loss: 0.0661
  Batch 100/831 - Loss: 0.0676
  Batch 110/831 - Loss: 0.0590
  Batch 120/831 - Loss: 0.0579
  Batch 130/831 - Loss: 0.0518
  Batch 140/831 - Loss: 0.0587
  Batch 150/831 - Loss: 0.0518
  Batch 160/831 - Loss: 0.0548
  Batch 170/831 - Loss: 0.0424
  Batch 180/831 - Loss: 0.0526
  Batch 190/831 - Loss: 0.0434
  Batch 200/831 - Loss: 0.0332
  Batch 210/831 - Loss: 0.0328
  Batch 220/831 - Loss: 0.0361
  Batch 230/831 - Loss: 0.0427
  Batch 240/831 - Loss: 0.0554
  Batch 250/831 - Loss: 0.0258
  Batch 260/831 - Loss: 0.0336
  Batch 270/831 - Loss: 0.0288
  Batch 280/831 - Loss: 0.0468
  Batch 

In [13]:
torch.save(model.state_dict(), "deeplabv3_model_epoch1.pth")
